In [2]:
import os
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
model1 = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

model2 = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

In [18]:
class BlogState(TypedDict):
    topic: str
    outline: str
    blog_post: str
    rating: int

In [19]:
def generate_outline(state: BlogState) -> BlogState:
    prompt = f"Generate a detailed outline for a blog post on the topic: {state['topic']}"
    response = model2.invoke(prompt).content
    state['outline'] = response.strip()
    return state

def generate_blog_post(state: BlogState) -> BlogState:
    prompt = f"Write a blog post based on the following topic as well as outline:\n{state['topic']}\n{state['outline']}"
    response = model1.invoke(prompt).content
    state['blog_post'] = response.strip()
    return state

def evaluate_blog_post(state: BlogState) -> BlogState:
    prompt = f"Evaluate the following blog post and give it a rating (1-10):\n{state['blog_post']}\nPlease provide only the rating as an integer."
    response = model2.invoke(prompt).content
    state['rating'] = int(response.strip())
    return state

In [26]:
# create graph

graph = StateGraph(BlogState)

# add nodes

graph.add_node("generate_outline", generate_outline)
graph.add_node("generate_blog_post", generate_blog_post)
graph.add_node("evaluate_blog_post", evaluate_blog_post)

# add edges

graph.add_edge(START, "generate_outline")
graph.add_edge("generate_outline", "generate_blog_post")
graph.add_edge("generate_blog_post", "evaluate_blog_post")
graph.add_edge("evaluate_blog_post", END)

# compile graph

workflow = graph.compile()

In [27]:
final_state = workflow.invoke({"topic": "Future risk in Artificial Intelligence", "outline": "", "blog_post": "", "rating": 0})

In [30]:
print(final_state["topic"])
print("\n")
print(final_state["outline"])
print("\n")
print(final_state["blog_post"])
print("\n")
print(final_state["rating"])

Future risk in Artificial Intelligence


Here is a detailed outline for a blog post on the topic "Future Risk in Artificial Intelligence":

**I. Introduction**

* Brief overview of Artificial Intelligence (AI) and its growing presence in modern life
* Importance of considering the potential risks associated with AI
* Thesis statement: As AI continues to advance and become more integrated into our daily lives, it's essential to acknowledge and address the potential future risks associated with this technology.

**II. Current State of AI and its Applications**

* Overview of current AI applications in industries such as:
	+ Healthcare
	+ Finance
	+ Transportation
	+ Education
* Discussion of the benefits and limitations of current AI systems

**III. Potential Future Risks in AI**

* **Job Displacement and Economic Impact**
	+ Discussion of the potential for AI to automate jobs and displace human workers
	+ Examination of the economic implications of widespread job displacement
* **Bias a